# INR Mesh Deformation Trained by Lightcurve Fitting (Self-Supervised)

Pipeline — **no ground-truth 3D supervision anywhere** (no Chamfer, no vertex L2, no GT mesh metrics):

sphere/ellipsoid prior (fixed topology) → `LightcurveToMesh` INR → deformed vertices
→ differentiable facet geometry (normals / areas) → convex LS+Lambert forward model
→ normalized lightcurves → aligned MSE vs observed curves.

$$
\mathcal{L} \;=\; \mathrm{MSE}\big(\hat{y},\, y\big)
$$

Implementation notes

- The convex photometric forward matrix is **rebuilt from the deformed mesh every step** and remains fully differentiable through facet geometry and visibility clamps.
- Per-camera period-scale + circular-shift alignment (as in `forward_model.ipynb`) is re-estimated every `ALIGN_REFRESH_EVERY` steps under `no_grad` and applied differentiably.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from hac.constants import OBSERVERS
from hac.data import load_data
from hac.lc2mesh.data import create_ellipsoidal_prior_mesh
from hac.lc2mesh.forward_model import (
    AsteroidForwardModel,
    observers_to_unit_tensor,
)
from hac.lc2mesh.metrics import (
    apply_circular_shifts,
    apply_period_scales,
    compute_mse,
    compute_pearson_r,
    compute_per_camera_metrics,
    estimate_best_period_scales_and_shifts,
)
from hac.lc2mesh.model import LightcurveToMesh
from hac.lc2mesh.utils import (
    set_seed,
    setup_device,
    vertex_normals_from_faces,
)
from hac.lc2mesh.visualization import (
    plot_camera_grid,
    plot_interactive_mesh,
    plot_lightcurve_comparison,
)

torch.set_default_dtype(torch.float64)


In [ ]:
# ============================== CONFIG ==============================
# --- Data ---
ASTEROID_ID = 3
CURVE_TYPE = "intensity"
SEED = 42
TARGET_GPU_ID = 1
PRIOR_SHAPE_TYPE = "ellipsoid"  # "ellipsoid", "peanut", "cube", "diamond" or "irregular"

# --- Prior mesh: icosphere scaled to the challenge cylinder (R, R, 1) ---
N_SUBDIVISIONS = 4  # 4 -> 2562 vertices / 5120 faces

# --- INR deformer (LightcurveToMesh) ---
INR_HIDDEN_DIM = 256
INR_NUM_LAYERS = 8
MAX_RADIUS_PERTURB = 0.2
COARSE_W0 = 15.0
DETAIL_W0 = 30.0

# --- Convex forward model (photometric settings follow forward_model.ipynb) ---
LAMBERT_WEIGHT = 0.8  # winner of the Lambert grid search in forward_model.ipynb
PHASE_ALPHA0_DEG = 30.0
PHASE_K = 0.02

# --- Curve alignment ---
USE_PHASE_ALIGNMENT = False
PERIOD_SCALE_GRID = np.linspace(0.80, 1.20, 81)
ALIGN_REFRESH_EVERY = 10  # steps between per-camera scale/shift re-estimation

# --- Training (lightcurve MSE only; no 3D GT) ---
USE_PRIOR_EMBEDDING = False
TRAIN_STEPS = 2000
LR = 1e-3
PRINT_EVERY = 50

CHECKPOINT_DIR = Path("ckpt_with_prior_embedding" if USE_PRIOR_EMBEDDING else "ckpt_no_prior_embedding")
RECON_DIR = Path("recon_with_prior_embedding" if USE_PRIOR_EMBEDDING else "recon_no_prior_embedding")
CHECKPOINT_DIR.mkdir(exist_ok=True)
RECON_DIR.mkdir(exist_ok=True)

CHECKPOINT_PATH = CHECKPOINT_DIR / f"inr_forward_train_prior_{PRIOR_SHAPE_TYPE}_ast{ASTEROID_ID}.pth"
PRIOR_EMBEDDING_PATH = Path(f"phase1_prior_embedding.pth")
# =====================================================================

set_seed(SEED)
device = setup_device(TARGET_GPU_ID)

In [ ]:
# --- Observed lightcurves (the only training target) ---
camera_names, e_obs = observers_to_unit_tensor(OBSERVERS, device=device, dtype=torch.float64)
e_sun = torch.tensor([-1.0, 0.0, 0.0], dtype=torch.float64, device=device)

ds_real = load_data(ASTEROID_ID, origin="real", curve_type=CURVE_TYPE)
observed = torch.as_tensor(
    ds_real.data[camera_names].to_numpy(dtype=np.float64, copy=True).T,
    dtype=torch.float64,
    device=device,
)
observed_norm = observed / observed.mean(dim=1, keepdim=True).clamp_min(1e-12)
n_phase = observed.shape[1]
phases = torch.arange(n_phase, dtype=torch.float64, device=device) * (2.0 * torch.pi / n_phase)

# --- Sphere/ellipsoid prior with fixed topology (same builder as priormesh.ipynb) ---
prior = create_ellipsoidal_prior_mesh(
    asteroid_id=ASTEROID_ID, subdivisions=N_SUBDIVISIONS, device=device, shape_type=PRIOR_SHAPE_TYPE
)
base_vertices = prior["prior_vertices_batch"].to(dtype=torch.float64)  # [1, V, 3]
faces = prior["prior_faces_t"]                                         # [F, 3]
n_vertices = int(base_vertices.shape[1])

# True vertex normals of the (fixed) prior mesh: deformation directions for the INR.
base_normals = vertex_normals_from_faces(base_vertices, faces)  # [1, V, 3]

# --- load checkpoint metadata first so constructor hyperparams stay in sync ---
prior_embedding = None
if USE_PRIOR_EMBEDDING and PRIOR_EMBEDDING_PATH.exists():
    prior_embedding = torch.load(PRIOR_EMBEDDING_PATH, map_location=device)
    ckpt_cfg = prior_embedding.get("config", prior_embedding)
    INR_HIDDEN_DIM = int(ckpt_cfg.get("inr_hidden_dim", INR_HIDDEN_DIM))
    INR_NUM_LAYERS = int(ckpt_cfg.get("inr_num_layers", INR_NUM_LAYERS))
    MAX_RADIUS_PERTURB = float(ckpt_cfg.get("max_radius_perturb", MAX_RADIUS_PERTURB))
    COARSE_W0 = float(ckpt_cfg.get("coarse_w0", COARSE_W0))
    DETAIL_W0 = float(ckpt_cfg.get("detail_w0", DETAIL_W0))

# --- INR deformer: continuous radial field over the prior sphere directions ---
net = LightcurveToMesh(
    inr_hidden_dim=INR_HIDDEN_DIM,
    inr_num_layers=INR_NUM_LAYERS,
    coarse_w0=COARSE_W0,
    detail_w0=DETAIL_W0,
    max_radius_perturb=MAX_RADIUS_PERTURB,
).to(device)
n_params = sum(p.numel() for p in net.parameters())

if prior_embedding is not None:
    state = prior_embedding.get("state_dict", prior_embedding.get("model_state"))
    if state is None:
        raise KeyError("Checkpoint missing both 'state_dict' and 'model_state'.")
    net.load_state_dict(state)
    print(f"Loaded prior embedding from {PRIOR_EMBEDDING_PATH}")
    print(
        f"Checkpoint hyperparams: hidden={INR_HIDDEN_DIM}, layers={INR_NUM_LAYERS}, "
        f"max_radius={MAX_RADIUS_PERTURB}, coarse_w0={COARSE_W0}, detail_w0={DETAIL_W0}"
    )

print(f"Asteroid {ASTEROID_ID}: cameras={len(camera_names)}, phases={n_phase}")
print(
    f"Prior mesh: {n_vertices} vertices, {int(faces.shape[0])} faces "
    f"(cylinder R={prior['R_CYLINDER']}) \n"
    f"Prior mesh extents: "
    f"x[{base_vertices[0, :, 0].min():.3f}, {base_vertices[0, :, 0].max():.3f}] "
    f"y[{base_vertices[0, :, 1].min():.3f}, {base_vertices[0, :, 1].max():.3f}] "
    f"z[{base_vertices[0, :, 2].min():.3f}, {base_vertices[0, :, 2].max():.3f}]"

)
print(f"INR parameters: {n_params:,}")

In [ ]:
# --- Test the net on the prior mesh (no training yet) and visualize the output ---
warm_started = bool(USE_PRIOR_EMBEDDING and PRIOR_EMBEDDING_PATH.exists())

net.eval()
with torch.no_grad():
    deformed_init, residual_init = net(base_vertices, base_normals)
net.train()

init_vertices_np = base_vertices[0].detach().cpu().numpy()
disp = residual_init[0].norm(dim=1)

assert deformed_init.shape == base_vertices.shape
assert residual_init.shape == base_vertices.shape
print(f"Deformed vertices: {tuple(deformed_init.shape)} (matches prior {tuple(base_vertices.shape)})")
print(
    f"Normal displacement |r|: mean={float(disp.mean()):.5f}, max={float(disp.max()):.5f} "
    f"| all finite: {bool(torch.isfinite(deformed_init).all())}"
)
if warm_started:
    print("State: warm-started from the phase-1 prior embedding -> should already resemble the asteroid.")
else:
    print("State: untrained (zero-init output heads) -> displacement ~0, mesh equals the ellipsoidal prior.")

fig_init = plot_interactive_mesh(
    vertices=init_vertices_np,
    faces=faces.detach().cpu().numpy(),
    title=(
        f"Asteroid {ASTEROID_ID}: INR output BEFORE lightcurve training "
        f"({'phase-1 warm start' if warm_started else 'untrained'}, {int(faces.shape[0])} faces)"
    ),
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_init.show()

fig_deformed = plot_interactive_mesh(
    vertices=deformed_init[0].detach().cpu().numpy(),
    faces=faces.detach().cpu().numpy(),
    title=(
        f"Asteroid {ASTEROID_ID}: INR output BEFORE lightcurve training "
        f"({'phase-1 warm start' if warm_started else 'untrained'}, {int(faces.shape[0])} faces)"
    ),
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_deformed.show()


In [ ]:
# --- Differentiable convex mesh -> lightcurve pipeline + loss ---
R_CYL = float(prior["R_CYLINDER"])


def facet_geometry(vertices: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Facet normals [F,3] and areas [F] from vertices [V,3] (differentiable)."""
    tri = vertices[faces]  # [F, 3, 3]
    edge1 = tri[:, 1] - tri[:, 0]
    edge2 = tri[:, 2] - tri[:, 0]
    cross = torch.cross(edge1, edge2, dim=1)
    twice_area = cross.norm(dim=1)
    normals = cross / twice_area.clamp_min(1e-12).unsqueeze(1)
    return normals, 0.5 * twice_area


def canonicalize_to_cylinder(vertices: torch.Tensor, R: float = R_CYL) -> torch.Tensor:
    """Differentiably gauge-fix vertices [V,3] to the challenge cylinder D(0,R) x [-1,1].

    Enforces the three exact facts from the challenge metadata:
    - containment: x^2 + y^2 <= R^2, |z| <= 1
    - touching:    min(z) = -1 and max(z) = +1 (object meets both planes)
    - tightness:   max(sqrt(x^2 + y^2)) = R (R is the minimal radius)

    Differentiable analogue of normalize_mesh_to_challenge_cylinder
    (center_xy="bounds"). Mean-normalized lightcurves cannot see uniform
    scale, so this also removes the scale gauge freedom during training.
    """
    z = vertices[:, 2]
    z_mid = 0.5 * (z.max() + z.min())
    z_half = (0.5 * (z.max() - z.min())).clamp_min(1e-9)
    xy = vertices[:, :2]
    xy_mid = 0.5 * (xy.max(dim=0).values + xy.min(dim=0).values)
    xy_c = xy - xy_mid
    s_xy = R / xy_c.norm(dim=1).max().clamp_min(1e-9)
    return torch.cat([xy_c * s_xy, ((z - z_mid) / z_half).unsqueeze(1)], dim=1)


def predicted_lightcurves(vertices: torch.Tensor) -> torch.Tensor:
    """Deformed mesh -> normalized convex lightcurves [C, P] (differentiable)."""
    normals, areas = facet_geometry(vertices)
    fm = AsteroidForwardModel(
        normals=normals,
        lambert_weight=LAMBERT_WEIGHT,
        alpha0=float(np.deg2rad(PHASE_ALPHA0_DEG)),
        k_phase=PHASE_K,
    )
    A = fm.build_forward_matrix(phases=phases, e_sun=e_sun, e_obs=e_obs)  # [C, P, F]
    raw = torch.einsum("cpf,f->cp", A, areas)
    return raw / raw.mean(dim=1, keepdim=True).clamp_min(1e-12)


def align_prediction(
    pred: torch.Tensor,
    period_scales: torch.Tensor | None,
    phase_shifts: torch.Tensor | None,
) -> torch.Tensor:
    """Per-camera period-scale resample + circular shift (differentiable in pred)."""
    out = pred
    if period_scales is not None:
        out = apply_period_scales(out, period_scales)
    if phase_shifts is not None:
        out = apply_circular_shifts(out, phase_shifts)
    return out


# --- Sanity check: gradients flow end-to-end from lightcurve MSE into the INR ---
deformed0, _ = net(base_vertices, base_normals)
verts0 = canonicalize_to_cylinder(deformed0[0])
pred0 = predicted_lightcurves(verts0)
loss0 = compute_mse(observed_norm, pred0)
loss0.backward()

grad_norm = torch.sqrt(
    sum((p.grad**2).sum() for p in net.parameters() if p.grad is not None)
)
all_finite = all(torch.isfinite(p.grad).all() for p in net.parameters() if p.grad is not None)
print(f"Sanity check: loss={float(loss0.item()):.6f}, INR grad_norm={float(grad_norm):.3e}, all_finite={bool(all_finite)}")
print(
    f"Canonicalized extents: z[{float(verts0.detach()[:, 2].min()):.3f}, {float(verts0.detach()[:, 2].max()):.3f}], "
    f"max_r={float(verts0.detach()[:, :2].norm(dim=1).max()):.3f} (target R={R_CYL})"
)

net.zero_grad(set_to_none=True)
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# --- Training loop: convex lightcurve MSE only (no 3D GT) ---
optimizer = torch.optim.SGD(net.parameters(), lr=LR, momentum=0.9, nesterov=True)
loss_fn = torch.nn.MSELoss()

history: dict[str, list[float]] = {
    "step": [], "loss": [], "mean_r": [], "min_r": [],
}
period_scales = None
phase_shifts = None
t_start = time.time()

for step in range(1, TRAIN_STEPS + 1):
    optimizer.zero_grad()
    deformed, _ = net(base_vertices, base_normals)
    pred = predicted_lightcurves(deformed[0])

    # Re-estimate per-camera alignment periodically (no_grad), apply differentiably.
    if USE_PHASE_ALIGNMENT and (step == 1 or step % ALIGN_REFRESH_EVERY == 0):
        period_scales, phase_shifts, _ = estimate_best_period_scales_and_shifts(
            observed_norm, pred.detach(), PERIOD_SCALE_GRID
        )
    if USE_PHASE_ALIGNMENT:
        pred = align_prediction(pred, period_scales, phase_shifts)

    loss = loss_fn(observed_norm, pred)

    loss.backward()
    optimizer.step()

    with torch.no_grad():
        r_cam = compute_pearson_r(observed_norm, pred, dim=1, eps=1e-12)

    history["step"].append(step)
    history["loss"].append(float(loss.item()))
    history["mean_r"].append(float(r_cam.mean().item()))
    history["min_r"].append(float(r_cam.min().item()))

    if step == 1 or step % PRINT_EVERY == 0:
        print(
            f"step {step:4d} | loss={history['loss'][-1]:.6f} "
            f"| mean_r={history['mean_r'][-1]:.4f} | min_r={history['min_r'][-1]:.4f}"
        )

elapsed = time.time() - t_start
model_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
torch.save(
    {
        "state_dict": model_state,
        "model_state": model_state,
        "config": {
            "asteroid_id": ASTEROID_ID,
            "n_subdivisions": N_SUBDIVISIONS,
            "inr_hidden_dim": INR_HIDDEN_DIM,
            "inr_num_layers": INR_NUM_LAYERS,
            "max_radius_perturb": MAX_RADIUS_PERTURB,
            "coarse_w0": COARSE_W0,
            "detail_w0": DETAIL_W0,
            "lambert_weight": LAMBERT_WEIGHT,
        },
    },
    CHECKPOINT_PATH,
)
print(
    f"Training done in {elapsed / 60.0:.1f} min | checkpoint: {CHECKPOINT_PATH}"
)

In [ ]:
# --- Training curves: loss + correlation ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["step"], history["loss"], label="Training LC MSE", color="#d95f02")
axes[0].set_yscale("log")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("MSE")
axes[0].set_title("Training MSE")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history["step"], history["mean_r"], label="Mean per-camera r", color="#d95f02")
axes[1].plot(history["step"], history["min_r"], label="Min per-camera r", color="#1b9e77")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Pearson r")
axes[1].set_title("Lightcurve Correlation")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# --- Final evaluation: direct forward-model lightcurves ---
with torch.no_grad():
    deformed_final, _ = net(base_vertices, base_normals)
    pred_norm = predicted_lightcurves(deformed_final[0])

per_camera_df = compute_per_camera_metrics(observed_norm, pred_norm, camera_names)
print(
    f"Final lightcurve fit: mean_r={per_camera_df['Pearson_R'].mean():.4f}, "
    f"min_r={per_camera_df['Pearson_R'].min():.4f}, "
    f"mean MSE={per_camera_df['MSE'].mean():.6f}"
)
display(per_camera_df.sort_values("Pearson_R"))

target_cam = "135_hor1"
fig, _ = plot_lightcurve_comparison(
    obs_norm=observed_norm.detach().cpu().numpy(),
    pred_norm=pred_norm.detach().cpu().numpy(),
    camera_keys=camera_names,
    target_cam_id=target_cam,
)
plt.show()

phase_deg = np.rad2deg(phases.detach().cpu().numpy())
fig, _ = plot_camera_grid(observed_norm, pred_norm, camera_names, phase_axis=phase_deg)
plt.show()

In [ ]:
from hac.data import get_model_folder
from hac.mesh import normalize_mesh_to_challenge_cylinder
import trimesh  # Commonly used for point-in-mesh checks


GT_EXIST = True
model_dir = get_model_folder(ASTEROID_ID)
stl_file = model_dir / f"asteroid{ASTEROID_ID}.stl"
if not stl_file.exists():
    GT_EXIST = False
    print(f"Warning: GT STL file not found at {stl_file} -> skipping voxel measure calculation.")
else:
    # Raw challenge STLs span [-3, 3]^3, but the whole training pipeline lives in
    # the normalized cylinder frame D(0, R) x [-1, 1]. Bring GT into the same
    # frame before comparing extents or computing the voxel measure.
    true_mesh = normalize_mesh_to_challenge_cylinder(
        trimesh.load_mesh(stl_file), prior["R_CYLINDER"]
    )

# --- Interactive 3D view of the INR-deformed mesh (same utility as forward_model.ipynb) ---
vertices_np = deformed_final[0].detach().cpu().numpy()
faces_np = faces.detach().cpu().numpy()

fig_mesh = plot_interactive_mesh(
    vertices=vertices_np,
    faces=faces_np,
    title=(
        f"Asteroid {ASTEROID_ID}: INR-deformed mesh ({faces_np.shape[0]} faces) | "
        f"mean_r={per_camera_df['Pearson_R'].mean():.3f}"
    ),
    mesh_color="#c0b08a",
    show_edges=False,
)
fig_mesh.show()

if GT_EXIST:
    gt_fig_mesh = plot_interactive_mesh(
        vertices=true_mesh.vertices,
        faces=true_mesh.faces,
        title=f"Asteroid {ASTEROID_ID}: GT mesh, cylinder-normalized ({true_mesh.faces.shape[0]} faces)",
        mesh_color="#a0c0ff",
        show_edges=False,
    )
    gt_fig_mesh.show()

print(
    f"Deformed mesh extents: "
    f"x[{vertices_np[:, 0].min():.3f}, {vertices_np[:, 0].max():.3f}] "
    f"y[{vertices_np[:, 1].min():.3f}, {vertices_np[:, 1].max():.3f}] "
    f"z[{vertices_np[:, 2].min():.3f}, {vertices_np[:, 2].max():.3f}] "
    f"(cylinder bound: R={prior['R_CYLINDER']}, z in [-1, 1])"
)
if GT_EXIST:
    print(
        f"GT mesh extents (cylinder-normalized): "
        f"x[{true_mesh.vertices[:, 0].min():.3f}, {true_mesh.vertices[:, 0].max():.3f}] "
        f"y[{true_mesh.vertices[:, 1].min():.3f}, {true_mesh.vertices[:, 1].max():.3f}] "
        f"z[{true_mesh.vertices[:, 2].min():.3f}, {true_mesh.vertices[:, 2].max():.3f}]"
    )

# --- Save the final INR-deformed mesh as STL ---
recon_mesh = trimesh.Trimesh(vertices=vertices_np, faces=faces_np, process=False)
recon_mesh.export(RECON_DIR / f"prior_{PRIOR_SHAPE_TYPE}_asteroid{ASTEROID_ID}.stl")
print(f"Saved INR-deformed mesh as STL: {RECON_DIR / f'prior_{PRIOR_SHAPE_TYPE}_asteroid{ASTEROID_ID}.stl'}")

In [ ]:
# --- Evaluation voxel measure calculation ---
from hac.eval import calculate_voxel_measure

if GT_EXIST:
    # compute the voxel measure between the predicted mesh and the ground truth mesh
    pred_mesh = trimesh.Trimesh(vertices=vertices_np, faces=faces_np, process=False)
    score = calculate_voxel_measure(true_mesh, pred_mesh)
    print(f"Voxel measure between predicted mesh and GT mesh: {score:.6f}")
else:
    print("GT mesh not available -> skipping voxel measure calculation.")